In [3]:
import torch
from torch import nn

In [4]:
# We define a helper function to calculate convolutions. It initializes the
# convolutional layer weights and performs corresponding dimensionality
# elevations and reductions on the input and output
def comp_conv2d(conv2d, X):
    # (1, 1) indicates that batch size and the number of channels are both 1
    X = X.reshape((1, 1) + X.shape)
    Y = conv2d(X)
    # Strip the first two dimensions: examples and channels
    return Y.reshape(Y.shape[2:])

# 1 row and column is padded on either side, so a total of 2 rows or columns
# are added
conv2d = nn.LazyConv2d(1, kernel_size=3, padding=1)
X = torch.rand(size=(8, 8))
comp_conv2d(conv2d, X).shape


torch.Size([8, 8])

In [5]:
# We use a convolution kernel with height 5 and width 3. The padding on either
# side of the height and width are 2 and 1, respectively
conv2d = nn.LazyConv2d(1, kernel_size=(5, 3), padding=(2, 1))
comp_conv2d(conv2d, X).shape


torch.Size([8, 8])

In [6]:
conv2d = nn.LazyConv2d(1, kernel_size=3, padding=1, stride=2)
comp_conv2d(conv2d, X).shape


torch.Size([4, 4])

In [7]:
conv2d = nn.LazyConv2d(1, kernel_size=(3, 5), padding=(0, 1), stride=(3, 4))
comp_conv2d(conv2d, X).shape



torch.Size([2, 2])

# Multiple Input Channels

In [8]:
def corr2d(X, K):
    Y = torch.zeros((
        X.shape[0] - K.shape[0] + 1,
        X.shape[1] - K.shape[1] + 1
    ))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i,j] = ( X[i:i+K.shape[0] , j:j + K.shape[1]] * K ).sum()

    return Y;


# wrote this from scratch
''' 3 mistakes, 
1. Wrong shape of Y, use this formula Yi = Xi - Ki + 1
2. for iteration use limits of Y as ultimately im storing the data in Y.
3. .sum() and sum() are different. sum() adds rows and will give a vector output
    but i need a scalar output for my feature_map and .sum() does it.
'''

' 3 mistakes, \n1. Wrong shape of Y, use this formula Yi = Xi - Ki + 1\n2. for iteration use limits of Y as ultimately im storing the data in Y.\n3. .sum() and sum() are different. sum() adds rows and will give a vector output\n    but i need a scalar output for my feature_map and .sum() does it.\n'

In [9]:
def corr2d_multi_in(X, K):
    return sum(corr2d(x, k) for x, k in zip(X, K))

In [10]:
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

corr2d_multi_in(X, K)


tensor([[ 56.,  72.],
        [104., 120.]])

In [11]:
# this time we are preserving corr output of each channels

def corr2d_multi_in_out(X, K):
    return torch.stack( [corr2d_multi_in(X,k) for k in K],0)

In [12]:
K = torch.stack( (K, K + 1, K + 2), 0)
K.shape

torch.Size([3, 2, 2, 2])

In [13]:
corr2d_multi_in_out(X, K)

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

In [16]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape( ( c_i , h*w))
    K = K.reshape((c_o , c_i))

    Y = torch.matmul(K, X)
    return Y.reshape((c_o, h, w))

In [17]:
X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))
Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6
